In [21]:
%pip install pixeltable-pgserver
%pip install psycopg

In [ ]:
import pixeltable_pgserver as pgserver
server = pgserver.get_server('/tmp/my_db_data')
db_url = server.get_uri()
print("Посилання для %sql:", db_url)

Посилання для %sql: postgresql://postgres:@/postgres?host=/tmp/my_db_data


In [ ]:
%load_ext sql
%sql postgresql://postgres:@/postgres?host=/tmp/my_db_data
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
%%sql
SELECT version();

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
1 rows affected.


version
"PostgreSQL 16.15 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit"


# 1. Завантажуємо дані з файла infectious_cases.csv у таблицю

In [33]:
import os
import io
import pandas as pd
from sqlalchemy import create_engine, text
import pixeltable_pgserver as pgserver
from google.colab import drive
from google.colab import data_table
drive.mount('/content/drive')

# 1. Автоматично підхоплюємо URI вбудованого в Colab pgserver
server = pgserver.get_server('/tmp/my_db_data')
db_url = server.get_uri()
engine = create_engine(db_url)

# 2. Шлях до файлу на Google Диску
FILE_PATH = '/content/drive/MyDrive/Postgres SQL/infectious_cases.csv'

# 3. Створення схеми pandemic та її активація
with engine.connect() as connection:
    # Створюємо схему, якщо вона ще не існує
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS pandemic;"))
    # Робимо її схемою за замовчуванням для цієї сесії
    connection.execute(text("SET search_path TO pandemic;"))
    connection.commit()
    print("Схему 'pandemic' успішно створено та обрано.")

# 4. Зчитування даних з CSV за допомогою pandas
print("Зчитування CSV файлу...")
df = pd.read_csv(FILE_PATH)

# Включаємо покращене відображення для всіх таблиць pandas
data_table.enable_dataframe_formatter()

# Завантажуємо перші 100 рядків для перегляду
with engine.connect() as connection:
    df_preview = pd.read_sql("SELECT * FROM pandemic.infectious_cases LIMIT 100;", connection)
    df_count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM pandemic.infectious_cases;", connection)

# Виводимо загальну кількість рядків
print(f"\nЗагальна кількість рядків у таблиці: {df_count['total_rows'].values[0]}")
print("\nІнтерактивна таблиця для ознайомлення (перші 100 рядків):")
df_preview

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Схему 'pandemic' успішно створено та обрано.
Зчитування CSV файлу...

Загальна кількість рядків у таблиці: 10521

Інтерактивна таблиця для ознайомлення (перші 100 рядків):


,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,None,6160.0,0.0,NaN,NaN,NaN,NaN,None,NaN
1,Afghanistan,AFG,1981,None,5859.0,0.0,NaN,NaN,NaN,NaN,None,NaN
2,Afghanistan,AFG,1982,None,9730.0,0.0,NaN,NaN,NaN,NaN,None,NaN
3,Afghanistan,AFG,1983,None,13937.0,0.0,NaN,NaN,NaN,NaN,None,NaN
4,Afghanistan,AFG,1984,None,3864.0,0.0,NaN,NaN,NaN,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Africa,None,1979,None,NaN,NaN,NaN,NaN,NaN,NaN,None,21586.0
96,Africa,None,2021,None,NaN,NaN,NaN,NaN,NaN,NaN,None,141467.0
97,African Region (WHO),None,1990,None,NaN,NaN,4283.0730,161375200.0,1716678.8,1419726.5,None,NaN
98,African Region (WHO),None,1991,None,NaN,NaN,4313.5760,165035680.0,1980984.6,1434700.4,None,NaN


# 2. Нормалізуємо таблицю infectious_cases до 3-ї нормальної форми

In [74]:
%%sql
-- 1. Встановлюємо схему за замовченням (чтобы не писать pandemic. перед каждой таблицей)
SET search_path TO pandemic;

-- 2. Створюємо таблицю-довідник країн
CREATE TABLE IF NOT EXISTS countries (
    id SERIAL PRIMARY KEY,
    entity VARCHAR(255) NOT NULL UNIQUE,
    code VARCHAR(10)
);

-- 3. Заповнюємо довідник унікальними країнами і кодами з початкової таблиці
INSERT INTO countries (entity, code)
SELECT DISTINCT entity, code
FROM infectious_cases
ON CONFLICT (entity) DO NOTHING;

-- 4. Створюємо нову нормалізовану таблицю захворювань із зовнішним ключом
CREATE TABLE IF NOT EXISTS infectious_cases_normalized (
    id SERIAL PRIMARY KEY,
    country_id INT NOT NULL,
    year INT NOT NULL,
    number_yaws NUMERIC,
    polio_cases NUMERIC,
    cases_guinea_worm NUMERIC,
    number_rabies NUMERIC,
    number_malaria NUMERIC,
    number_hiv NUMERIC,
    number_tuberculosis NUMERIC,
    number_smallpox NUMERIC,
    number_cholera_cases NUMERIC,
    FOREIGN KEY (country_id) REFERENCES countries(id) ON DELETE CASCADE
);

-- 5. Переносимо дані, з'єднауючи їх через JOIN з довідником для отримання id
INSERT INTO infectious_cases_normalized (
    country_id, year, number_yaws, polio_cases, cases_guinea_worm,
    number_rabies, number_malaria, number_hiv, number_tuberculosis,
    number_smallpox, number_cholera_cases
)
SELECT
    c.id,
    i.year,
    i.number_yaws,
    i.polio_cases,
    i.cases_guinea_worm,
    i.number_rabies,
    i.number_malaria,
    i.number_hiv,
    i.number_tuberculosis,
    i.number_smallpox,
    i.number_cholera_cases
FROM infectious_cases i
JOIN countries c ON i.entity = c.entity;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
Done.
0 rows affected.
Done.
10521 rows affected.


[]

Перевіряємо навність створеної першої таблиці

In [44]:
%%sql
SELECT * FROM countries LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
10 rows affected.


id,entity,code
1,Russia,RUS
2,Trinidad and Tobago,TTO
3,Antigua and Barbuda,ATG
4,Nicaragua,NIC
5,Denmark,DNK
6,Turkey,TUR
7,Sudan,SDN
8,Myanmar,MMR
9,Macao,MAC
10,Togo,TGO


Перевіряємо навність створеної другої таблиці

In [42]:
%%sql
SELECT * FROM infectious_cases_normalized LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
10 rows affected.


id,country_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
1,212,1980,None,6160,0,None,None,None,None,None,None
2,212,1981,None,5859,0,None,None,None,None,None,None
3,212,1982,None,9730,0,None,None,None,None,None,None
4,212,1983,None,13937,0,None,None,None,None,None,None
5,212,1984,None,3864,0,None,None,None,None,None,None
6,212,1985,None,13867,0,None,None,None,None,None,None
7,212,1986,None,12901,0,None,None,None,None,None,None
8,212,1987,None,4396,0,None,None,None,None,None,None
9,212,1988,None,2149,0,None,None,None,None,None,None
10,212,1989,None,385,0,None,None,None,None,None,None


# 3. Аналізуємо дані таблиць:
Для атрибута Number_rabies шукаємо середнє значення, мінімальне значення, максимальне значення, суму. Сортуємо за порахованим середнім значенням у порядку спадання та виводимо перші 10 рядків

In [46]:
%%sql
SET search_path TO pandemic;

SELECT
    c.id AS country_id,
    c.entity,
    c.code,
    AVG(icn.number_rabies) AS avg_rabies,
    MIN(icn.number_rabies) AS min_rabies,
    MAX(icn.number_rabies) AS max_rabies,
    SUM(icn.number_rabies) AS total_rabies
FROM infectious_cases_normalized icn
JOIN countries c ON icn.country_id = c.id
WHERE icn.number_rabies IS NOT NULL -- Виключаємо країни, де взагалі немає даних про rabies
GROUP BY c.id, c.entity, c.code
ORDER BY avg_rabies DESC
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
10 rows affected.


country_id,entity,code,avg_rabies,min_rabies,max_rabies,total_rabies
84,World,OWID_WRL,20192.370366666667,14075.508,24744.658,2423084.444
44,Lower Middle Income (WB),None,15193.959383333333,10202.528,19182.795,1823275.1260
158,South Asia (WB),None,11729.888753333333,7271.2754,15361.878,1407586.6504
139,South-East Asia Region (WHO),None,11424.326893333333,6806.0127,15641.958,1370919.2272
66,G20,None,10189.046003333333,6339.0796,13164.881,1222685.5204
223,India,IND,8599.1732800000000000,5425.8726,11121.142,1031900.7936
119,Sub-Saharan Africa (WB),None,4748.2680900000000000,4356.8184,5017.977,569792.1708
77,African Region (WHO),None,4649.7023766666666667,4283.073,4917.784,557964.2852
193,Low Income (WB),None,3568.1054500000000000,3014.617,3873.533,428172.6540
178,East Asia & Pacific (WB),None,3427.4611333333333333,2009.9395,4591.0107,411295.3360


In [51]:
%%sql
SET search_path TO pandemic;

SELECT
    id,
    year,
    -- 1. Створюємо дату першого січня відповідного року
    MAKE_DATE(year, 1, 1) AS first_year_date,

    -- 2. Отримуємо поточну дату системи
    CURRENT_DATE AS current_query_date,

    -- 3. Рахуємо точну різницю в роках між поточною датою та 1 січня
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1))) AS years_diff
FROM infectious_cases_normalized
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
20 rows affected.


id,year,first_year_date,current_query_date,years_diff
1,1980,1980-01-01,2026-11-07,46
2,1981,1981-01-01,2026-11-07,45
3,1982,1982-01-01,2026-11-07,44
4,1983,1983-01-01,2026-11-07,43
5,1984,1984-01-01,2026-11-07,42
6,1985,1985-01-01,2026-11-07,41
7,1986,1986-01-01,2026-11-07,40
8,1987,1987-01-01,2026-11-07,39
9,1988,1988-01-01,2026-11-07,38
10,1989,1989-01-01,2026-11-07,37


# 5. Створюємо власну функцію

5.1. Створюємо функцію для обчислення різниці в роках

In [52]:
%%sql
SET search_path TO pandemic;

CREATE OR REPLACE FUNCTION calculate_years_difference(input_year INT)
RETURNS INT AS $$
BEGIN
    RETURN EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(input_year, 1, 1)));
END;
$$ LANGUAGE plpgsql;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
Done.


[]

5.2. Створюємо функцію для розрахунку захворювань за період

In [53]:
%%sql
SET search_path TO pandemic;

CREATE OR REPLACE FUNCTION calculate_cases_per_period(cases NUMERIC, divider NUMERIC)
RETURNS NUMERIC AS $$
BEGIN
    IF divider = 0 THEN
        RETURN NULL;
    END IF;
    RETURN cases / divider;
END;
$$ LANGUAGE plpgsql;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
Done.


[]

5.3. Робимо запит для першої функції (різниця в роках)

In [55]:
%%sql
SET search_path TO pandemic;

SELECT
    id,
    year,
    calculate_years_difference(year) AS years_ago
FROM infectious_cases_normalized
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
10 rows affected.


id,year,years_ago
1,1980,46
2,1981,45
3,1982,44
4,1983,43
5,1984,42
6,1985,41
7,1986,40
8,1987,39
9,1988,38
10,1989,37


5.4. Робимо запит для другої функції (розрахунок захворювань за періодами)

In [70]:
%%sql
SET search_path TO pandemic;

SELECT
    id,
    country_id,
    year,
    number_malaria AS yearly_malaria,
    -- Середня кількість на місяць (/12)
    ROUND(calculate_cases_per_period(number_malaria, 12),2) AS monthly_avg_malaria,
    -- Середня кількість на квартал (/4)
    ROUND(calculate_cases_per_period(number_malaria, 4),2) AS quarterly_avg_malaria,
    -- Середня кількість на півріччя (/2)
    ROUND(calculate_cases_per_period(number_malaria, 2),2) AS half_year_avg_malaria
FROM infectious_cases_normalized
WHERE number_malaria IS NOT NULL
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
10 rows affected.


id,country_id,year,yearly_malaria,monthly_avg_malaria,quarterly_avg_malaria,half_year_avg_malaria
11,212,1990,1219845.8,101653.82,304961.45,609922.90
12,212,1991,1132781.5,94398.46,283195.38,566390.75
13,212,1995,725758.3,60479.86,181439.58,362879.15
14,212,1997,605853.06,50487.76,151463.27,302926.53
15,212,1998,547045.4,45587.12,136761.35,273522.70
16,212,1999,502898.88,41908.24,125724.72,251449.44
17,212,2000,491614.72,40967.89,122903.68,245807.36
18,212,2001,511626.2,42635.52,127906.55,255813.10
19,212,2002,546030.6,45502.55,136507.65,273015.30
20,212,2003,574740.3,47895.03,143685.08,287370.15


In [73]:
%%sql
SET search_path TO pandemic;

SELECT
    icn.id,
    c.entity AS country,
    c.code AS country_code,
    icn.year,
    icn.number_tuberculosis AS yearly_tuberculosis,
    ROUND(calculate_cases_per_period(icn.number_tuberculosis, 12), 2) AS monthly_avg_tuberculosis,
    ROUND(calculate_cases_per_period(icn.number_tuberculosis, 4), 2) AS quarterly_avg_tuberculosis,
    ROUND(calculate_cases_per_period(icn.number_tuberculosis, 2), 2) AS half_year_avg_tuberculosis
FROM infectious_cases_normalized icn
JOIN countries c ON icn.country_id = c.id
WHERE icn.number_tuberculosis IS NOT NULL
ORDER BY icn.number_tuberculosis DESC -- Спочатку виводимо найгірші показники
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fmy_db_data
Done.
10 rows affected.


id,country,country_code,year,yearly_tuberculosis,monthly_avg_tuberculosis,quarterly_avg_tuberculosis,half_year_avg_tuberculosis
41868,World,OWID_WRL,2002,8957511,746459.25,2239377.75,4478755.50
20826,World,OWID_WRL,2002,8957511,746459.25,2239377.75,4478755.50
31347,World,OWID_WRL,2002,8957511,746459.25,2239377.75,4478755.50
10305,World,OWID_WRL,2002,8957511,746459.25,2239377.75,4478755.50
20829,World,OWID_WRL,2005,8955053,746254.42,2238763.25,4477526.50
10308,World,OWID_WRL,2005,8955053,746254.42,2238763.25,4477526.50
31350,World,OWID_WRL,2005,8955053,746254.42,2238763.25,4477526.50
41871,World,OWID_WRL,2005,8955053,746254.42,2238763.25,4477526.50
20827,World,OWID_WRL,2003,8954095,746174.58,2238523.75,4477047.50
10306,World,OWID_WRL,2003,8954095,746174.58,2238523.75,4477047.50
